In [1]:
from data_prep import *
from xgboost_regress import *

In [2]:
import os
import pandas as pd

parent_dat = None
for folder in os.listdir('data'):
    for file in os.listdir(os.path.join('data', folder)):
        if file.endswith('.csv'):
            dat = pd.read_csv(os.path.join('data', folder, file))
            if parent_dat is None:
                parent_dat = dat
            else:
                parent_dat = pd.concat([parent_dat, dat], ignore_index=True)

prepped_data = prep_data(parent_dat).dropna()

prepped_data

,BEGIN_LAT,BEGIN_LON,STORM_AREA_SQMILES,DURATION_MINUTES,TEMPERATURE_F,ANOMALY_F,RONI_AVG,POPULATION,MEDIAN_INCOME,MEDIAN_YEAR_BUILT,...,MODAL_YEAR_BUILT_BIN_built_1970_to_1979,MODAL_YEAR_BUILT_BIN_built_1980_to_1989,MODAL_YEAR_BUILT_BIN_built_1990_to_1999,MODAL_YEAR_BUILT_BIN_built_2000_to_2009,MODAL_YEAR_BUILT_BIN_built_2010_to_2019,MODAL_YEAR_BUILT_BIN_built_2020_or_later,COASTAL_TYPE_SHORELINE_inland,COASTAL_TYPE_SHORELINE_shoreline,COASTAL_TYPE_WATERSHED_inland,COASTAL_TYPE_WATERSHED_watershed
4,42.5589,-92.5583,21.113817,854,37.6,4.9,0.600000,14892.0,47702.0,1951.0,...,0,0,0,0,0,0,1,0,1,0
10,33.9420,-81.9282,0.000000,2,63.7,0.5,-1.700000,26680.0,42834.0,1984.0,...,0,0,0,1,0,0,1,0,1,0
21,38.0594,-78.4837,0.000000,0,76.6,5.7,-0.933333,96633.0,64847.0,1987.0,...,0,0,0,1,0,0,1,0,1,0
22,38.0302,-78.4853,0.000000,0,76.6,5.7,-0.933333,96633.0,64847.0,1987.0,...,0,0,0,1,0,0,1,0,1,0
24,39.1753,-77.9090,0.000000,0,78.2,4.1,-1.266667,14013.0,73244.0,1973.0,...,0,0,0,0,0,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
544762,40.3894,-79.8080,1.472851,3,71.5,3.1,-0.433333,1238177.0,78548.0,1957.0,...,0,0,0,0,0,0,1,0,1,0
544763,40.4269,-80.3060,0.029395,1,70.7,2.7,-0.433333,210042.0,78958.0,1966.0,...,1,0,0,0,0,0,1,0,1,0
544766,40.1754,-81.8500,0.000000,0,70.9,2.6,-0.433333,36744.0,55577.0,1967.0,...,1,0,0,0,0,0,1,0,1,0
544772,39.7769,-80.1302,0.001539,2,70.0,1.9,-0.433333,34835.0,68041.0,1960.0,...,1,0,0,0,0,0,1,0,1,0


In [3]:
X = prepped_data.drop(columns=['log10_damage'])
y = prepped_data['log10_damage']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=8)

search = tune_model(X_train, X_test, y_train, y_test, n_iter=50)

c:\miniconda\envs\damage\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:08:41] WARNING: C:\Users\task_177740979073858\croot\xgboost-split_1777409985184\work\src\context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


CV RMSE: 0.6091
best params: {'colsample_bytree': np.float64(0.876201218034711), 'gamma': np.float64(0.31625638277172186), 'learning_rate': np.float64(0.037245155512243826), 'max_depth': 8, 'min_child_weight': 3, 'n_estimators': 784, 'reg_alpha': np.float64(0.5456957318755907), 'reg_lambda': np.float64(3.453872469197429), 'subsample': np.float64(0.8937150517432543)}
Train RMSE: 0.5108
Test RMSE: 0.6051
Test R2:   0.5145


In [ ]:
import matplotlib.pyplot as plt
# show the feature importance
imp = pd.Series(
    search.best_estimator_.get_booster().get_score(importance_type="gain")
).sort_values(ascending=False)

imp_collapsed = imp.copy()
for prefix in ["EVENT_TYPE_", "MODAL_YEAR_BUILT_BIN_",
               "COASTAL_TYPE_SHORELINE_", "COASTAL_TYPE_WATERSHED_"]:
    cols = [c for c in imp.index if c.startswith(prefix)]
    imp_collapsed[prefix.rstrip("_")] = imp_collapsed[cols].sum()
    imp_collapsed = imp_collapsed.drop(cols)
imp_collapsed = imp_collapsed.sort_values(ascending=False)

imp_collapsed.plot(y = imp_collapsed.index, x = imp_collapsed.values, kind="bar")
plt.title("Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    search.best_estimator_, X_test, y_test,
    n_repeats=10, random_state=8, n_jobs=-1,
    scoring="neg_root_mean_squared_error",
)

perm_imp = pd.Series(
    result.importances_mean, index=X_test.columns,
).sort_values(ascending=False)

perm_imp_collapsed = perm_imp.copy()
for prefix in ["EVENT_TYPE_", "MODAL_YEAR_BUILT_BIN_",
               "COASTAL_TYPE_SHORELINE_", "COASTAL_TYPE_WATERSHED_"]:
    cols = [c for c in perm_imp.index if c.startswith(prefix)]
    perm_imp_collapsed[prefix.rstrip("_")] = perm_imp_collapsed[cols].sum()
    perm_imp_collapsed = perm_imp_collapsed.drop(cols)

perm_imp_collapsed = perm_imp_collapsed.sort_values(ascending=False)

perm_imp_collapsed.plot(y = perm_imp_collapsed.index, x = perm_imp_collapsed.values, kind="bar")
plt.title("Permutation Feature Importance")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
import shap

explainer = shap.TreeExplainer(search.best_estimator_)
sv = explainer.shap_values(X_test)
shap.summary_plot(sv, X_test, max_display=20)   # bar + bee-swarm
shap.dependence_plot("STORM_AREA_SQMILES", sv, X_test)